In [4]:
import torch
import torchvision.transforms as transforms
import torchvision.transforms.functional as F
import torch.nn as nn
import torch.nn.functional as F_nn
import torch.optim as optim
import numpy as np
from sklearn.datasets import fetch_lfw_people
from torch.utils.data import Dataset, DataLoader
import random
import sqlite3
import pickle
from PIL import Image

In [5]:
print("Downloading dataset via scikit-learn...")
lfw = fetch_lfw_people(min_faces_per_person = 5, color = True, resize=1.0, slice_=None)
images = lfw.images
labels = lfw.target
names = lfw.target_names
print(f"Successfully loaded {len(images)} images!")
print(f"Raw Numpy Shape: {images.shape} (N, H, W, C)")

Successfully loaded 5985 images!
Raw Numpy Shape: (5985, 250, 250, 3) (N, H, W, C)


In [6]:
images_transposed = np.transpose(images, (0, 3, 1, 2))

tensor_images = torch.tensor(images_transposed).float()

tensor_images = F.resize(tensor_images, size=[128, 128])

print(f"Final PyTorch Tensor Shape: {tensor_images.shape}")


Final PyTorch Tensor Shape: torch.Size([5985, 3, 128, 128])


In [7]:
class TripleFaceDataset(Dataset):
    def __init__(self, tensor_images, labels):
        self.images = tensor_images
        self.labels = labels
        self.unique_labels = np.unique(labels)
        # Label mapped with respective indices
        self.label_to_indices = {label: np.where(self.labels == label)[0] for label in self.unique_labels}
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, index):
        anchor_img = self.images[index]
        anchor_label = self.labels[index]

        positive_idx = random.choice(self.label_to_indices[anchor_label])
        positive_img = self.images[positive_idx]

        negative_label = random.choice(labels)
        while(negative_label == anchor_label):
            negative_label = random.choice(labels)
        
        negative_idx = random.choice(self.label_to_indices[negative_label])
        negative_img = self.images[negative_idx]

        return anchor_img, positive_img, negative_img



In [8]:
dataset = TripleFaceDataset(tensor_images, labels)

dataloader = DataLoader(dataset, batch_size=32, shuffle=True, drop_last=True)

print(f"Total Batches ready for training: {len(dataloader)}")

anchor_batch, positive_batch, negative_batch = next(iter(dataloader))
print(f"One Anchor Batch Shape: {anchor_batch.shape}")

Total Batches ready for training: 187
One Anchor Batch Shape: torch.Size([32, 3, 128, 128])


In [9]:
class FaceEmbeddingNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels = 3, out_channels = 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels = 32, out_channels = 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(in_channels = 64, out_channels = 128, kernel_size=3, padding=1)

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        self.fc = nn.Linear(in_features = 128*16*16, out_features = 128)
        
    def forward(self, x):

        x = self.pool(F_nn.relu(self.conv1(x)))
        x = self.pool(F_nn.relu(self.conv2(x)))
        x = self.pool(F_nn.relu(self.conv3(x)))

        x = torch.flatten(x, start_dim=1)

        x = self.fc(x)

        x = F_nn.normalize(x, p=2, dim=1)

        return x

print("FaceNet Architecture Ready")

        

FaceNet Architecture Ready


In [10]:
model = FaceEmbeddingNet()

criterion = nn.TripletMarginLoss(margin = 1.0, p = 2)

optimizer = optim.Adam(model.parameters(), lr=0.001)

In [14]:
epochs = 5

for epoch in range(epochs):
    total_loss = 0.0

    for batch_idx, (anchor, positive, negative) in enumerate(dataloader):
        optimizer.zero_grad()

        emb_anchor = model(anchor)
        emb_positive = model(positive)
        emb_negative = model(negative)

        loss = criterion(emb_anchor, emb_positive, emb_negative)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss/len(dataloader)
    print(f"Epoch {epoch+1}/{epochs} - Average Loss: {avg_loss:.4f}")

print("Training Finished! Your model can now recognize faces!")

KeyboardInterrupt: 

In [10]:
torch.save(model.state_dict(), "face_model.pth")
print("Model saved successfully")

Model saved successfully


In [11]:
# I am aiming to make it scalable in future by shifting from sqlite3 to a more robust database solution like PostgreSQL or MongoDB, but for now, let's set up our SQLite database to store face embeddings.
conn = sqlite3.connect("face_database.db")
cursor = conn.cursor()

cursor.execute("""
    CREATE TABLE IF NOT EXISTS face_embeddings (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL,
            embedding BLOB NOT NULL
    )
""")

conn.commit()
print("Database ready!")

Database ready!


In [ ]:
def register_face(name, image_tensors, model, conn):
    model.eval()

    if not isinstance(image_tensors, list):
        image_tensors = [image_tensors]
    
    all_embeddings = []

    with torch.no_grad():
        for image in image_tensors:
            current_image = image.unsqueeze(0)
            embedding = model(current_image)
            all_embeddings.append(embedding)
    
    stacked = torch.stack(all_embeddings)
    average_embeddings = stacked.mean(dim = 0)

    average_embeddings = F_nn.normalize(average_embeddings, p=2, dim=1)

    embedding_bytes = pickle.dumps(average_embeddings)
    cursor = conn.cursor()
    cursor.execute("INSERT INTO face_embeddings (name, embedding) VALUES (?, ?)", (name, embedding_bytes))
    conn.commit()

    print(f"Registered: {name} (using {len(image_tensors)} image(s))")


In [1]:
def recognize_face(image_tensor, model, conn, threshold=0.8):
    model.eval()
    with torch.no_grad():
        batched_image = image_tensor.unsqueeze(0)
        new_embedding = model(batched_image)
    
    cursor = conn.cursor()
    cursor.execute("SELECT name, embedding FROM face_embeddings")
    rows = cursor.fetchall()

    if len(rows) == 0:
        return "No faces registered yet!"

    best_match = "Unknown"
    best_distance = float('inf')

    for name, embedding_bytes in rows:
        stored_embedding = pickle.loads(embedding_bytes)

        distance = torch.dist(new_embedding, stored_embedding).item()

        if distance < best_distance:
            best_match = name
            best_distance = distance

    if best_distance > threshold:
        return f"Unknown (closest was {best_match} at distance {best_distance:.4f})"
    
    return f"Match: {best_match} (distance: {best_distance:.4f})"

In [13]:
model = FaceEmbeddingNet()
model.load_state_dict(torch.load("face_model.pth"))
model.eval()

photo_transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor()
])

img1 = Image.open(r"C:\Users\adty2\OneDrive\Pictures\PICTURES\WhatsApp Image 2025-04-18 at 18.24.01_9f2d194a.jpg")
img2 = Image.open(r"C:\Users\adty2\OneDrive\Pictures\PICTURES\WhatsApp Image 2025-04-18 at 18.24.01_caa9c50d.jpg")

tensor_img1 = photo_transform(img1)
tensor_img2 = photo_transform(img2)

register_face("Me", tensor_img1, model, conn)

result = recognize_face(tensor_img2, model, conn)
print(f"\nRecognition Result: {result}")


NameError: name 'register_face' is not defined